# KAE Remote GPU Worker: vLLM OpenAI-Compatible API Server

Данный скрипт запускает **vLLM** с моделью `Qwen/Qwen2.5-7B-Instruct` (или аналогичной) в средах **Google Colab** с T4/L4/A100 GPU и предоставляет OpenAI-совместимый API эндпоинт через **ngrok** / **localtunnel** для подключения к **Knowledge Assembly Engine (KAE) HybridLLMRouter**.

In [ ]:
# 1. Установка зависимостей vLLM и прокси
!pip install -q vllm fastapi uvicorn pyngrok nest_asyncio

import os
import subprocess
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

# 2. Авторизация ngrok (опционально, вставьте токен со страницы https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "" 
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 3. Конфигурация модели и запуск vLLM
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
print(f"🚀 Запуск сервера vLLM для модели: {MODEL_NAME}...")

cmd = f"python3 -m vllm.entrypoints.openai.api_server --model {MODEL_NAME} --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.90"
process = subprocess.Popen(cmd, shell=True)

# 4. Проброс туннеля ngrok
http_tunnel = ngrok.connect(8000)
print(f"\n=======================================================")
print(f"✅ ВАШ УДАЛЕННЫЙ KAE COLAB ENDPOINT:")
print(f"🔗 {http_tunnel.public_url}/v1")
print(f"Используйте этот URL в KAE .env файле как AI_BASE_URL:")
print(f"AI_BASE_URL={http_tunnel.public_url}/v1")
print(f"AI_MODEL={MODEL_NAME}")
print(f"=======================================================\n")